In [0]:
from pyspark.sql import functions as F


# ============================================================
# 1. Configuration
# ============================================================

CATALOG = "dbx_lyonloyer_dev"

SOURCE_TABLE = f"{CATALOG}.silver.dvf"
TARGET_TABLE = f"{CATALOG}.gold.dvf_gold_appartement"


# ============================================================
# 2. Read Silver
# ============================================================

df = spark.table(SOURCE_TABLE)

print("Silver rows:", df.count())

In [0]:
# garder que appartment
df_apt = df.filter(
    (F.col("nb_apartments") == 1) &
    (F.col("nb_houses") == 0) &
    (F.col("surface_apartment") > 0) &
    (F.col("sale_price") > 0)
)

print("Appartement rows:", df_apt.count())

In [0]:
# avoir nb de pièce
df_apt = df_apt.withColumn(
    "rooms",
    F.when(F.col("nb_apt_1_room") == 1, 1)
     .when(F.col("nb_apt_2_rooms") == 1, 2)
     .when(F.col("nb_apt_3_rooms") == 1, 3)
     .when(F.col("nb_apt_4_rooms") == 1, 4)
     .when(F.col("nb_apt_5_rooms") == 1, 5)
     .otherwise(None)
)

df_apt = df_apt.withColumn(
    "room_category",
    F.when(F.col("rooms") == 1, "T1")
     .when(F.col("rooms") == 2, "T2")
     .when(F.col("rooms") == 3, "T3")
     .when(F.col("rooms") == 4, "T4")
     .when(F.col("rooms") == 5, "T5")
     .otherwise(None)
)

In [0]:
# calcule prix surface
df_apt = (
    df_apt
    .withColumn(
        "surface_m2",
        F.col("surface_apartment").cast("double")
    )
    .withColumn(
        "price_per_m2",
        F.round(
            F.col("sale_price") / F.col("surface_m2"),
            2
        )
    )
)

In [0]:
# ============================================================
# Extreme low prices
# ============================================================

display(
    df_apt.filter(F.col("price_per_m2") < 1000)
          .select(
              "mutation_id",
              "mutation_date",
              "insee_code",
              "lyon_arrondissement",
              "room_category",
              "sale_price",
              "surface_m2",
              "price_per_m2",
              "vefa"
          )
          .orderBy("price_per_m2")
          .limit(100)
)

In [0]:
# ============================================================
# Filter unrealistic price per m²
# ============================================================
df_apt_clean = df_apt.filter(
    (F.col("surface_m2") >= 10) &
    (F.col("surface_m2") <= 500)
)

df_apt_clean = df_apt_clean.filter(
    (F.col("price_per_m2") >= 1000) &
    (F.col("price_per_m2") <= 15000)
)

In [0]:
# table gold final
df_gold = df_apt_clean.select(
    "mutation_id",
    "mutation_date",
    "year",
    "insee_code",
    "lyon_arrondissement",

    "property_type",
    "room_category",
    "rooms",

    "sale_price",
    "surface_m2",
    "price_per_m2",

    "vefa",
    "nb_lots"
)

In [0]:
display(df_gold.limit(20))

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("dbx_lyonloyer_dev.gold.dvf")